In [1]:
#LOAD STATEMENTS
import torch
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
import warnings
warnings.filterwarnings('ignore', '.*do not.*', )
warnings.warn('Do not show this message')
import matplotlib.pyplot as plt
import os
import torch

In [2]:

#multimodal imports
import torch
from dataloader import train_loader, val_loader, test_loader
from dataloader import df_train, df_val, df_test
import numpy as np
import pandas as pd

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2072 entries, 0 to 2071
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             2072 non-null   object 
 1   pCR             1432 non-null   float64
 2   n_xy            2072 non-null   float64
 3   n_z             2072 non-null   float64
 4   n_times         2072 non-null   float64
 5   pre             2072 non-null   float64
 6   post_early      2072 non-null   float64
 7   post_late       2072 non-null   float64
 8   slice_thick     2072 non-null   float64
 9   xy_spacing      2072 non-null   float64
 10  mask_start      2072 non-null   float64
 11  mask_end        2072 non-null   float64
 12  sraw            2072 non-null   float64
 13  eraw            2072 non-null   float64
 14  scol            2072 non-null   float64
 15  ecol            2072 non-null   float64
 16  tum_vol         2071 non-null   float64
 17  age             2072 non-null   f

In [3]:
from encoder import SmallCNNEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----- LOAD MODEL -----

model = SmallCNNEncoder(in_ch=9, feat_dim=256)
model.load_state_dict(torch.load("encoder_ssl_3.pth", map_location=device))
model = model.to(device)
model.eval()

for p in model.parameters():
    p.requires_grad = False


C:\Users\Ivón\AppData\Local\Temp\ipykernel_28812\1461430249.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("encoder_ssl_3.pth", map_loc

In [4]:
print(model.eval())

SmallCNNEncoder(
  (net): Sequential(
    (0): Conv2d(9, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=1)
  )
  (fc): Linear(in_features=128, out_features=256, bias=True)
)


In [5]:
@torch.no_grad()
def extract_embeddings(model, dl, device):
    model.eval()
    pid_to_embs = {}

    for xb, _, pid in dl:
        xb = xb.to(device)

        emb = model(xb).cpu().numpy()  

        for p, e in zip(pid, emb):
            pid_to_embs.setdefault(p, []).append(e)

    # agregamos por paciente (mean)
    pids = []
    embs = []
    for p in pid_to_embs:
        pids.append(p)
        embs.append(np.mean(pid_to_embs[p], axis=0))

    return pids, np.vstack(embs)


### Función para extraer EMBEDDINGS (256D)

In [6]:
pids_tr_ssl, emb_tr_ssl = extract_embeddings(model, train_loader, device)


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ISPY2-500826
['ISPY2-500826_spy2_vis1_dce_aqc_0.nii.gz', 'ISPY2-500826_spy2_vis1_dce_aqc_2.nii.gz', 'ISPY2-500826_spy2_vis1_dce_aqc_6.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-771151
['ACRIN-6698-771151_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-771151_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-771151_spy2_vis1_dce_aqc_6.nii.gz']
['ACRIN-6698-102212_

In [9]:
pids_va_ssl, emb_va_ssl = extract_embeddings(model, val_loader, device)


['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-414844
['ACRIN-6698-414844_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-414844_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-414844_spy2_vis1_dce_aqc_5.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-573605
['ACRIN-6698-573605_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-573605_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-573605_spy2_vis1_dce_aqc_6.nii.gz']


In [10]:
pids_te_ssl, emb_te_ssl = extract_embeddings(model, test_loader, device)

['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-102212
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz']
['ACRIN-6698-102212_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-102212_spy2_vis1_dce_aqc_6.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-103939_spy2_vis1_dce_aqc_5.nii.gz', 'ACRIN-6698-104268_spy2_vis1_dce_aqc_0.nii.gz'] ACRIN-6698-116603
['ACRIN-6698-116603_spy2_vis1_dce_aqc_0.nii.gz', 'ACRIN-6698-116603_spy2_vis1_dce_aqc_2.nii.gz', 'ACRIN-6698-116603_spy2_vis1_dce_aqc_6.nii.gz']


### Convertir embeddings a DataFrame

In [11]:
def emb_to_df(pids, emb, prefix="img"):
    df = pd.DataFrame(emb, columns=[f"{prefix}_{i}" for i in range(emb.shape[1])])
    df["pid"] = pids
    return df

df_emb_tr_ssl = emb_to_df(pids_tr_ssl, emb_tr_ssl)
df_emb_va_ssl = emb_to_df(pids_va_ssl, emb_va_ssl)
df_emb_te_ssl = emb_to_df(pids_te_ssl, emb_te_ssl)
df_emb_tr_ssl.to_csv("results/ssl_mri_embeddings_train.csv", index=False)
df_emb_va_ssl.to_csv("results/ssl_mri_embeddings_val.csv", index=False) 
df_emb_te_ssl.to_csv("results/ssl_mri_embeddings_test.csv", index=False)
df_emb_tr_ssl.head()


,img_0,img_1,img_2,img_3,img_4,img_5,img_6,img_7,img_8,img_9,...,img_247,img_248,img_249,img_250,img_251,img_252,img_253,img_254,img_255,pid
0,-0.002389,-0.012940,0.027030,-0.068002,0.142376,-0.057070,0.163638,0.025153,0.150893,0.016543,...,0.000407,0.191351,-0.261929,-0.069268,0.059842,0.063778,0.091114,-0.036523,0.045702,ISPY2-500826
1,0.024044,0.024770,0.053221,-0.054530,0.142996,-0.056103,0.147736,0.004879,0.071883,0.032031,...,0.043982,0.169207,-0.226632,-0.038406,-0.000372,0.068439,0.083686,-0.025131,0.017915,ACRIN-6698-771151
2,-0.006589,-0.024308,0.024484,-0.074113,0.149236,-0.062575,0.175391,0.031571,0.172200,0.017590,...,-0.008258,0.202224,-0.278025,-0.074405,0.073175,0.061571,0.096405,-0.039738,0.055517,ACRIN-6698-323721
3,0.015737,0.024676,0.046715,-0.044283,0.126462,-0.043100,0.121808,0.002983,0.065856,0.020273,...,0.037163,0.151051,-0.212648,-0.040936,0.004993,0.071626,0.082624,-0.028819,0.019523,ISPY2-353318
4,0.035468,0.085439,0.028454,-0.004237,0.078054,-0.012587,0.046643,-0.005359,-0.017060,-0.008388,...,0.070995,0.081955,-0.115327,-0.017614,-0.037282,0.072567,0.062551,-0.004404,-0.000150,ISPY2-798604


In [ ]:
features = [
    "age",
    "menopause",
    "tum_vol",
    "HR",
    "HER2",
    "TripleNeg"
]

df_cl_tr_ssl = df_train[["pid", "pCR"] + features]
df_cl_va_ssl = df_val[["pid", "pCR"] + features]
df_cl_te_ssl = df_test[["pid", "pCR"] + features]


# FUSIÓN MULTIMODAL

In [ ]:
df_mm_tr_ssl = df_cl_tr_ssl.merge(df_emb_tr_ssl, on="pid")
df_mm_va_ssl = df_cl_va_ssl.merge(df_emb_va_ssl, on="pid")
df_mm_te_ssl = df_cl_te_ssl.merge(df_emb_te_ssl, on="pid")


### Train / Val / Test para LR multimodal

In [ ]:
print("Original:", df_mm_tr_ssl.shape, df_mm_te_ssl.shape, df_mm_va_ssl.shape)
df_mm_tr_ssl = df_mm_tr_ssl.dropna()
df_mm_te_ssl = df_mm_te_ssl.dropna()
df_mm_va_ssl = df_mm_va_ssl.dropna()

print("After NaN filter:", df_mm_tr_ssl.shape, df_mm_te_ssl.shape, df_mm_va_ssl.shape)

Original: (784, 264) (99, 264) (99, 264)
After NaN filter: (754, 264) (96, 264) (94, 264)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X_tr = df_mm_tr_ssl.drop(columns=["pid", "pCR"])
y_tr = df_mm_tr_ssl["pCR"]

X_va = df_mm_va_ssl.drop(columns=["pid", "pCR"])
y_va = df_mm_va_ssl["pCR"]
X_te = df_mm_te_ssl.drop(columns=["pid", "pCR"])
y_te = df_mm_te_ssl["pCR"]

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

# X_img_train: (N_train, D) embeddings
# X_img_val:   (N_val, D)
# X_img_test:  (N_test, D)

sc_img = StandardScaler()
X_tr = sc_img.fit_transform(X_tr)      # fit SOLO train
X_va = sc_img.transform(X_va)
X_te = sc_img.transform(X_te)

pca = PCA(n_components=32, random_state=0)   # prueba 16, 32, 64
Xtr_p = pca.fit_transform(X_tr)               # fit SOLO train
Xva_p = pca.transform(X_va)
Xte_p = pca.transform(X_te)

print("Explained variance:", pca.explained_variance_ratio_.sum())


Explained variance: 0.9999852659341858


C:\Users\Ivón\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\Ivón\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


### Train

In [ ]:
lr_mm = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

lr_mm.fit(X_tr, y_tr)


C:\Users\Ivón\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

def find_best_alpha(y_val, p_clin_val, p_img_val):
    best_a, best_auc = None, -1
    for a in np.linspace(0, 1, 101):
        p = a*p_clin_val + (1-a)*p_img_val
        auc = roc_auc_score(y_val, p)
        if auc > best_auc:
            best_auc, best_a = auc, a
    return best_a, best_auc


In [ ]:
p_final_test = best_a*p_clin_test + (1-best_a)*p_img_test
auc_test = roc_auc_score(y_test, p_final_test)

print("Test AUC ensemble:", auc_test)


# Final evaluation

In [ ]:
probs_te = lr_mm.predict_proba(X_te)[:, 1]
auc_te = roc_auc_score(y_te, probs_te)

print("Multimodal AUC (test):", auc_te)
print(classification_report(y_te, probs_te > 0.5))

Multimodal AUC (test): 0.705050505050505
              precision    recall  f1-score   support

         0.0       0.80      0.62      0.70        66
         1.0       0.44      0.67      0.53        30

    accuracy                           0.64        96
   macro avg       0.62      0.64      0.62        96
weighted avg       0.69      0.64      0.65        96



# Logistic Regression Unimodal

In [ ]:

df_train = df_train.dropna()
df_val   = df_val.dropna()
df_test  = df_test.dropna()

X_train = df_train[features]
y_train = df_train["pCR"]

X_val   = df_val[features]
y_val   = df_val["pCR"]

X_test  = df_test[features]
y_test  = df_test["pCR"]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)

print("AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]))


AUC: 0.7247474747474747


In [2]:
import numpy as np

def auc_ci(auc, n_pos, n_neg):
    Q1 = auc / (2 - auc)
    Q2 = 2 * auc**2 / (1 + auc)
    se = np.sqrt(
        (auc * (1 - auc) +
         (n_pos - 1) * (Q1 - auc**2) +
         (n_neg - 1) * (Q2 - auc**2))
        / (n_pos * n_neg)
    )
    lower = auc - 1.96 * se
    upper = auc + 1.96 * se
    return lower, upper, se
ci_clin = auc_ci(0.725, 30, 66)
print("sin SSL 95% CI:", ci_clin[:2])

sin SSL 95% CI: (np.float64(0.6093536252345085), np.float64(0.8406463747654914))


In [3]:
ci_cnn = auc_ci(0.623, 30, 66)
print("sin SSL 95% CI:",   ci_cnn[:2], "AUROC:", 0.623,)

sin SSL 95% CI: (np.float64(0.498727014761125), np.float64(0.747272985238875)) AUROC: 0.623
